# set up

In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

In [4]:
import json
import re
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
from tqdm.auto import tqdm


import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

LABEL2ID = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


In [25]:
DATA_DIR = Path("/content/drive/MyDrive/NLP/data")
OUTPUT_DIR = Path("/content/drive/MyDrive/NLP/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

EVIDENCE_PATH = DATA_DIR / "evidence.json"
TRAIN_PATH = DATA_DIR / "train-claims.json"
DEV_PATH = DATA_DIR / "dev-claims.json"
TEST_PATH = DATA_DIR / "test-claims-unlabelled.json"

MODEL_NAME = "distilroberta-base"
MODEL_DIR = OUTPUT_DIR / "verifier_model"

DEV_PRED_PATH = OUTPUT_DIR / "dev_predictions_cnn_bilstm_multihead.json"
TEST_PRED_PATH = OUTPUT_DIR / "test_predictions_cnn_bilstm_multihead.json"

## set seed

In [6]:
import random
import numpy as np
import torch
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # For GPU
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Make Python hashing deterministic
    os.environ["PYTHONHASHSEED"] = str(seed)

    # Make CuDNN deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Optional: force deterministic algorithms
    # This can make training slower and may raise errors for some operations.
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(42)

## utility functions

In [7]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def normalise_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%°]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def simple_tokenise(text: str) -> List[str]:
    return normalise_text(text).split()


def get_claim_items(claims_json: Dict) -> List[Tuple[str, Dict]]:
    return list(claims_json.items())


def concatenate_evidence(
    evidence_ids: List[str],
    evidence_corpus: Dict[str, str],
    max_evidence: int = 5
) -> str:
    selected_ids = evidence_ids[:max_evidence]
    evidence_texts = []

    for eid in selected_ids:
        if eid in evidence_corpus:
            evidence_texts.append(evidence_corpus[eid])

    if len(evidence_texts) == 0:
        return "No relevant evidence found."

    return " ".join(evidence_texts)

## read data

In [8]:
evidence_corpus = load_json(EVIDENCE_PATH)
train_claims = load_json(TRAIN_PATH)
dev_claims = load_json(DEV_PATH)

print("Number of evidence passages:", len(evidence_corpus))
print("Number of train claims:", len(train_claims))
print("Number of dev claims:", len(dev_claims))

Number of evidence passages: 1208827
Number of train claims: 1228
Number of dev claims: 154


## build vocabulary

In [9]:
def build_vocab(claims_json, evidence_corpus, min_freq=2, max_vocab_size=50000):
    counter = Counter()

    for claim_id, instance in claims_json.items():
        claim_text = instance["claim_text"]
        counter.update(simple_tokenise(claim_text))

        for eid in instance.get("evidences", []):
            if eid in evidence_corpus:
                counter.update(simple_tokenise(evidence_corpus[eid]))

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1,
        "<CLAIM>": 2,
        "<EVIDENCE>": 3
    }

    for word, freq in counter.most_common(max_vocab_size):
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)

    return vocab


vocab = build_vocab(train_claims, evidence_corpus)
print("Vocab size:", len(vocab))# Retriever
class BM25Retriever:
    def __init__(self, evidence_corpus: Dict[str, str]):
        self.evidence_corpus = evidence_corpus
        self.evidence_ids = list(evidence_corpus.keys())
        self.evidence_texts = [evidence_corpus[eid] for eid in self.evidence_ids]

        print("Building BM25 index...")
        tokenised_corpus = [simple_tokenise(text) for text in tqdm(self.evidence_texts)]
        self.bm25 = BM25Okapi(tokenised_corpus)

    def retrieve(self, claim_text: str, top_k: int = 5) -> List[str]:
        query_tokens = simple_tokenise(claim_text)
        scores = self.bm25.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [self.evidence_ids[i] for i in top_indices]

    def evaluate_recall_at_k(self, claims_json: Dict, k: int = 5) -> float:
        total = 0
        hit = 0

        for claim_id, instance in tqdm(get_claim_items(claims_json)):
            claim_text = instance["claim_text"]
            gold_evidence = set(instance.get("evidences", []))

            if len(gold_evidence) == 0:
                continue

            retrieved = set(self.retrieve(claim_text, top_k=k))

            if len(gold_evidence.intersection(retrieved)) > 0:
                hit += 1

            total += 1

        return hit / total if total > 0 else 0.0


Vocab size: 6872


## reuse the retriever file

In [10]:
import json
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer


class LoadedFAISSRetriever:
    def __init__(
        self,
        evidence_corpus,
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        cache_dir="faiss_cache",
        device=None
    ):
        self.evidence_corpus = evidence_corpus
        self.model_name = model_name
        self.cache_dir = Path(cache_dir)

        safe_model_name = model_name.replace("/", "_")
        self.index_path = self.cache_dir / f"{safe_model_name}.faiss"
        self.ids_path = self.cache_dir / f"{safe_model_name}_evidence_ids.json"

        if not self.index_path.exists():
            raise FileNotFoundError(f"Cannot find FAISS index: {self.index_path}")

        if not self.ids_path.exists():
            raise FileNotFoundError(f"Cannot find evidence IDs: {self.ids_path}")

        print("Loading SentenceTransformer model...")
        self.model = SentenceTransformer(model_name, device=device)

        print("Loading FAISS index...")
        self.index = faiss.read_index(str(self.index_path))

        print("Loading evidence ID mapping...")
        with open(self.ids_path, "r", encoding="utf-8") as f:
            self.evidence_ids = json.load(f)

        if len(self.evidence_ids) != self.index.ntotal:
            raise ValueError(
                f"Mismatch: {len(self.evidence_ids)} evidence IDs but "
                f"{self.index.ntotal} FAISS vectors."
            )

        print("Retriever loaded successfully.")
        print("Number of indexed evidence passages:", self.index.ntotal)

    def retrieve(self, claim_text, top_k=5):
        claim_embedding = self.model.encode(
            [claim_text],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        scores, indices = self.index.search(claim_embedding, top_k)

        retrieved_ids = [self.evidence_ids[i] for i in indices[0]]
        return retrieved_ids

In [11]:
retriever = LoadedFAISSRetriever(
    evidence_corpus=evidence_corpus,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    cache_dir="/content/drive/MyDrive/NLP/outputs/faiss_cache",
    device=device
)

Loading SentenceTransformer model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading FAISS index...
Loading evidence ID mapping...
Retriever loaded successfully.
Number of indexed evidence passages: 1208827


# CNN+BiLSTM

## Encode text

In [12]:
def encode_tokens(tokens, vocab, max_len=512):
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    return ids[:max_len]


def encode_claim_evidence(claim_text, evidence_text, vocab, max_len=512):
    tokens = (
        ["<CLAIM>"]
        + simple_tokenise(claim_text)
        + ["<EVIDENCE>"]
        + simple_tokenise(evidence_text)
    )

    return encode_tokens(tokens, vocab, max_len=max_len)

## CNN + BiLSTM dataset

In [13]:
class CNNBiLSTMDataset(Dataset):
    def __init__(
        self,
        claims_json,
        evidence_corpus,
        vocab,
        max_len=512,
        max_evidence=5,
        use_gold_evidence=True,
        retriever=None,
        retrieval_top_k=5,
        is_test=False
    ):
        self.items = list(claims_json.items())
        self.evidence_corpus = evidence_corpus
        self.vocab = vocab
        self.max_len = max_len
        self.max_evidence = max_evidence
        self.use_gold_evidence = use_gold_evidence
        self.retriever = retriever
        self.retrieval_top_k = retrieval_top_k
        self.is_test = is_test

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        claim_id, instance = self.items[idx]
        claim_text = instance["claim_text"]

        if self.use_gold_evidence and not self.is_test:
            evidence_ids = instance.get("evidences", [])
        else:
            evidence_ids = self.retriever.retrieve(
                claim_text,
                top_k=self.retrieval_top_k
            )

        evidence_text = concatenate_evidence(
            evidence_ids=evidence_ids,
            evidence_corpus=self.evidence_corpus,
            max_evidence=self.max_evidence
        )

        input_ids = encode_claim_evidence(
            claim_text=claim_text,
            evidence_text=evidence_text,
            vocab=self.vocab,
            max_len=self.max_len
        )

        item = {
            "claim_id": claim_id,
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "evidence_ids": evidence_ids
        }

        if not self.is_test:
            item["label"] = torch.tensor(
                LABEL2ID[instance["claim_label"]],
                dtype=torch.long
            )

        return item


def cnn_bilstm_collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    attention_mask = (input_ids != vocab["<PAD>"]).long()

    output = {
        "claim_ids": [item["claim_id"] for item in batch],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "evidence_ids": [item["evidence_ids"] for item in batch]
    }

    if "label" in batch[0]:
        output["labels"] = torch.stack([item["label"] for item in batch])

    return output

## Model Design

In [21]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, hidden_states, attention_mask=None):
        scores = self.attention(hidden_states).squeeze(-1)

        if attention_mask is not None:
            attention_mask = attention_mask.bool()
            scores = scores.masked_fill(~attention_mask, -1e9)

        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(hidden_states * weights.unsqueeze(-1), dim=1)

        return pooled


class ImprovedCNNBiLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        dropout=0.3,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx
        )

        self.embedding_dropout = nn.Dropout(dropout)

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=cnn_channels,
                kernel_size=k,
                padding=k // 2
            )
            for k in kernel_sizes
        ])

        cnn_output_dim = cnn_channels * len(kernel_sizes)

        self.bilstm = nn.LSTM(
            input_size=cnn_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        lstm_output_dim = lstm_hidden_dim * 2

        self.attention_pooling = AttentionPooling(lstm_output_dim)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, num_labels)
        )

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        # [batch, seq_len, embedding_dim] -> [batch, embedding_dim, seq_len]
        conv_input = embedded.transpose(1, 2)

        conv_outputs = []

        for conv in self.convs:
            x = F.relu(conv(conv_input))
            conv_outputs.append(x)

        # [batch, cnn_channels * num_kernels, seq_len]
        conv_output = torch.cat(conv_outputs, dim=1)

        # [batch, seq_len, cnn_channels * num_kernels]
        lstm_input = conv_output.transpose(1, 2)

        lstm_output, _ = self.bilstm(lstm_input)

        pooled_output = self.attention_pooling(
            lstm_output,
            attention_mask=attention_mask
        )

        pooled_output = self.dropout(pooled_output)

        logits = self.classifier(pooled_output)

        return logits

## Evaluation function

In [22]:
def evaluate_cnn_bilstm(model, dataloader, device="cpu"):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    print(classification_report(
        all_labels,
        all_preds,
        target_names=[ID2LABEL[i] for i in range(4)]
    ))

    return acc, macro_f1

## Train CNN + BiLSTM

In [23]:
def train_cnn_bilstm_multikernel(
    train_claims,
    dev_claims,
    evidence_corpus,
    vocab,
    epochs=5,
    batch_size=32,
    lr=1e-3,
    max_len=256,
    max_evidence=4,
    device="cpu"
):
    set_seed(42)

    train_dataset = CNNBiLSTMDataset(
        claims_json=train_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    dev_dataset = CNNBiLSTMDataset(
        claims_json=dev_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    g = torch.Generator()
    g.manual_seed(42)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=cnn_bilstm_collate_fn,
        generator=g
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=cnn_bilstm_collate_fn
    )

    model = ImprovedCNNBiLSTMClassifier(
        vocab_size=len(vocab),
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        dropout=0.4,
        pad_idx=vocab["<PAD>"]
    ).to(device)

    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    criterion = nn.CrossEntropyLoss()

    best_macro_f1 = 0.0
    best_state_dict = None

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimiser.zero_grad()

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimiser.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print("Training loss:", round(avg_loss, 4))

        dev_acc, dev_macro_f1 = evaluate_cnn_bilstm(
            model,
            dev_loader,
            device
        )

        print("Dev accuracy with gold evidence:", round(dev_acc, 4))
        print("Dev macro F1 with gold evidence:", round(dev_macro_f1, 4))

        if dev_macro_f1 > best_macro_f1:
            best_macro_f1 = dev_macro_f1
            best_state_dict = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }
            print("New best model saved in memory.")

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model

In [24]:
cnn_bilstm_model = train_cnn_bilstm_multikernel(
    train_claims=train_claims,
    dev_claims=dev_claims,
    evidence_corpus=evidence_corpus,
    vocab=vocab,
    epochs=5,
    batch_size=32,
    lr=1e-3,
    max_len=256,
    max_evidence=4,
    device=device
)


Epoch 1/5


  0%|          | 0/39 [00:00<?, ?it/s]

Training loss: 1.2874


  0%|          | 0/5 [00:00<?, ?it/s]

                 precision    recall  f1-score   support

       SUPPORTS       0.44      1.00      0.61        68
        REFUTES       0.00      0.00      0.00        27
NOT_ENOUGH_INFO       0.00      0.00      0.00        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.44       154
      macro avg       0.11      0.25      0.15       154
   weighted avg       0.19      0.44      0.27       154

Dev accuracy with gold evidence: 0.4416
Dev macro F1 with gold evidence: 0.1532
New best model saved in memory.

Epoch 2/5


...

  0%|          | 0/39 [00:00<?, ?it/s]

Training loss: 1.093


  0%|          | 0/5 [00:00<?, ?it/s]

                 precision    recall  f1-score   support

       SUPPORTS       0.50      0.31      0.38        68
        REFUTES       0.00      0.00      0.00        27
NOT_ENOUGH_INFO       0.36      0.98      0.52        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.40       154
      macro avg       0.21      0.32      0.23       154
   weighted avg       0.32      0.40      0.31       154

Dev accuracy with gold evidence: 0.3961
Dev macro F1 with gold evidence: 0.2262


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Predict with retrieved evidence

In [27]:
def predict_claims_cnn_bilstm(
    claims_json,
    evidence_corpus,
    retriever,
    model,
    vocab,
    output_path,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=False,
    device="cpu"
):
    dataset = CNNBiLSTMDataset(
        claims_json=claims_json,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=False,
        retriever=retriever,
        retrieval_top_k=retrieval_top_k,
        is_test=is_test
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=cnn_bilstm_collate_fn
    )

    model.eval()
    predictions = {}

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            pred_ids = torch.argmax(logits, dim=1).cpu().numpy().tolist()

            for claim_id, pred_id, evidence_ids in zip(batch["claim_ids"], pred_ids, batch["evidence_ids"]):
              predictions[claim_id] = {
                  "claim_text": claims_json[claim_id]["claim_text"],
                  "claim_label": ID2LABEL[pred_id],
                  "evidences": evidence_ids[:max_evidence]
              }

    save_json(predictions, output_path)
    print("Saved predictions to:", output_path)

    return predictions

## Generate dev predictions

In [28]:
CNN_BILSTM_DEV_PRED_PATH = OUTPUT_DIR / "dev_predictions_cnn_bilstm_multihead_4evi.json"

dev_predictions_cnn_bilstm = predict_claims_cnn_bilstm(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model=cnn_bilstm_model,
    vocab=vocab,
    output_path=CNN_BILSTM_DEV_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=False,
    device=device
)

  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/dev_predictions_cnn_bilstm_multihead_4evi.json


## predict test claims

In [30]:
test_claims = load_json(TEST_PATH)

TEST_PRED_PATH = OUTPUT_DIR / "test_predictions_cnn_bilstm_multihead.json"

test_predictions = predict_claims_cnn_bilstm(
    claims_json=test_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model=cnn_bilstm_model,
    vocab=vocab,
    output_path=TEST_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=True,
    device=device
)

  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/test_predictions_cnn_bilstm_multihead.json


## eval.py command line

In [31]:
import os

# Change the current working directory to the NLP folder in Google Drive
os.chdir('/content/drive/MyDrive/NLP/')

In [32]:
!python eval.py --predictions outputs/dev_predictions_cnn_bilstm_multihead_4evi.json --groundtruth data/dev-claims.json


Evidence Retrieval F-score (F)    = 0.1509018759018759
Claim Classification Accuracy (A) = 0.42857142857142855
Harmonic Mean of F and A          = 0.2232103947848205


# CNN+BiLSTM - multihead

## Encode text

In [ ]:
def encode_tokens(tokens, vocab, max_len=512):
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    return ids[:max_len]


def encode_claim_evidence(claim_text, evidence_text, vocab, max_len=512):
    tokens = (
        ["<CLAIM>"]
        + simple_tokenise(claim_text)
        + ["<EVIDENCE>"]
        + simple_tokenise(evidence_text)
    )

    return encode_tokens(tokens, vocab, max_len=max_len)

## CNN + BiLSTM dataset

In [ ]:
class CNNBiLSTMDataset(Dataset):
    def __init__(
        self,
        claims_json,
        evidence_corpus,
        vocab,
        max_len=512,
        max_evidence=5,
        use_gold_evidence=True,
        retriever=None,
        retrieval_top_k=5,
        is_test=False
    ):
        self.items = list(claims_json.items())
        self.evidence_corpus = evidence_corpus
        self.vocab = vocab
        self.max_len = max_len
        self.max_evidence = max_evidence
        self.use_gold_evidence = use_gold_evidence
        self.retriever = retriever
        self.retrieval_top_k = retrieval_top_k
        self.is_test = is_test

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        claim_id, instance = self.items[idx]
        claim_text = instance["claim_text"]

        if self.use_gold_evidence and not self.is_test:
            evidence_ids = instance.get("evidences", [])
        else:
            evidence_ids = self.retriever.retrieve(
                claim_text,
                top_k=self.retrieval_top_k
            )

        evidence_text = concatenate_evidence(
            evidence_ids=evidence_ids,
            evidence_corpus=self.evidence_corpus,
            max_evidence=self.max_evidence
        )

        input_ids = encode_claim_evidence(
            claim_text=claim_text,
            evidence_text=evidence_text,
            vocab=self.vocab,
            max_len=self.max_len
        )

        item = {
            "claim_id": claim_id,
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "evidence_ids": evidence_ids
        }

        if not self.is_test:
            item["label"] = torch.tensor(
                LABEL2ID[instance["claim_label"]],
                dtype=torch.long
            )

        return item


def cnn_bilstm_collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    attention_mask = (input_ids != vocab["<PAD>"]).long()

    output = {
        "claim_ids": [item["claim_id"] for item in batch],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "evidence_ids": [item["evidence_ids"] for item in batch]
    }

    if "label" in batch[0]:
        output["labels"] = torch.stack([item["label"] for item in batch])

    return output

## Model Design

In [36]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, hidden_states, attention_mask=None):
        scores = self.attention(hidden_states).squeeze(-1)

        if attention_mask is not None:
            attention_mask = attention_mask.bool()
            scores = scores.masked_fill(~attention_mask, -1e9)

        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(hidden_states * weights.unsqueeze(-1), dim=1)

        return pooled


class CNNBiLSTMMultiheadClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx
        )

        self.embedding_dropout = nn.Dropout(dropout)

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=cnn_channels,
                kernel_size=k,
                padding=k // 2
            )
            for k in kernel_sizes
        ])

        cnn_output_dim = cnn_channels * len(kernel_sizes)

        self.bilstm = nn.LSTM(
            input_size=cnn_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        lstm_output_dim = lstm_hidden_dim * 2

        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=lstm_output_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True
        )
        self.layer_norm = nn.LayerNorm(lstm_output_dim)

        self.attention_pooling = AttentionPooling(lstm_output_dim)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, num_labels)
        )

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        # [batch, seq_len, embedding_dim] -> [batch, embedding_dim, seq_len]
        conv_input = embedded.transpose(1, 2)

        conv_outputs = []

        for conv in self.convs:
            x = F.relu(conv(conv_input))
            conv_outputs.append(x)

        # [batch, cnn_channels * num_kernels, seq_len]
        conv_output = torch.cat(conv_outputs, dim=1)

        # [batch, seq_len, cnn_channels * num_kernels]
        lstm_input = conv_output.transpose(1, 2)

        lstm_output, _ = self.bilstm(lstm_input)

        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = attention_mask == 0

        attn_output, _ = self.multihead_attn(
            query=lstm_output,
            key=lstm_output,
            value=lstm_output,
            key_padding_mask=key_padding_mask
        )

        attn_output = self.layer_norm(attn_output + lstm_output)

        pooled_output = self.attention_pooling(
            attn_output,
            attention_mask=attention_mask
        )

        pooled_output = self.dropout(pooled_output)

        logits = self.classifier(pooled_output)

        return logits

## Evaluation function

In [37]:
def evaluate_cnn_bilstm(model, dataloader, device="cpu"):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    print(classification_report(
        all_labels,
        all_preds,
        target_names=[ID2LABEL[i] for i in range(4)]
    ))

    return acc, macro_f1

## Train CNN + BiLSTM

In [38]:
def train_cnn_bilstm_multikernel_multihead(
    train_claims,
    dev_claims,
    evidence_corpus,
    vocab,
    epochs=5,
    batch_size=32,
    lr=1e-3,
    max_len=256,
    max_evidence=4,
    device="cpu"
):
    set_seed(42)

    train_dataset = CNNBiLSTMDataset(
        claims_json=train_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    dev_dataset = CNNBiLSTMDataset(
        claims_json=dev_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    g = torch.Generator()
    g.manual_seed(42)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=cnn_bilstm_collate_fn,
        generator=g
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=cnn_bilstm_collate_fn
    )

    model = CNNBiLSTMMultiheadClassifier(
        vocab_size=len(vocab),
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=vocab["<PAD>"]
    ).to(device)

    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    criterion = nn.CrossEntropyLoss()

    best_macro_f1 = 0.0
    best_state_dict = None

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimiser.zero_grad()

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimiser.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print("Training loss:", round(avg_loss, 4))

        dev_acc, dev_macro_f1 = evaluate_cnn_bilstm(
            model,
            dev_loader,
            device
        )

        print("Dev accuracy with gold evidence:", round(dev_acc, 4))
        print("Dev macro F1 with gold evidence:", round(dev_macro_f1, 4))

        if dev_macro_f1 > best_macro_f1:
            best_macro_f1 = dev_macro_f1
            best_state_dict = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }
            print("New best model saved in memory.")

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model

In [40]:
cnn_bilstm_multihead_model = train_cnn_bilstm_multikernel_multihead(
    train_claims=train_claims,
    dev_claims=dev_claims,
    evidence_corpus=evidence_corpus,
    vocab=vocab,
    epochs=5,
    batch_size=32,
    lr=1e-3,
    max_len=256,
    max_evidence=4,
    device=device
)


Epoch 1/5


  0%|          | 0/39 [00:00<?, ?it/s]

Training loss: 1.2957


  0%|          | 0/5 [00:00<?, ?it/s]

                 precision    recall  f1-score   support

       SUPPORTS       0.44      1.00      0.61        68
        REFUTES       0.00      0.00      0.00        27
NOT_ENOUGH_INFO       0.00      0.00      0.00        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.44       154
      macro avg       0.11      0.25      0.15       154
   weighted avg       0.19      0.44      0.27       154

Dev accuracy with gold evidence: 0.4416
Dev macro F1 with gold evidence: 0.1532
New best model saved in memory.

Epoch 2/5


...

  0%|          | 0/39 [00:00<?, ?it/s]

Training loss: 1.1311


  0%|          | 0/5 [00:00<?, ?it/s]

                 precision    recall  f1-score   support

       SUPPORTS       0.53      0.40      0.45        68
        REFUTES       0.00      0.00      0.00        27
NOT_ENOUGH_INFO       0.37      0.93      0.53        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.42       154
      macro avg       0.22      0.33      0.25       154
   weighted avg       0.33      0.42      0.34       154

Dev accuracy with gold evidence: 0.4221
Dev macro F1 with gold evidence: 0.2454


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Predict with retrieved evidence

In [41]:
def predict_claims_cnn_bilstm(
    claims_json,
    evidence_corpus,
    retriever,
    model,
    vocab,
    output_path,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=False,
    device="cpu"
):
    dataset = CNNBiLSTMDataset(
        claims_json=claims_json,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=False,
        retriever=retriever,
        retrieval_top_k=retrieval_top_k,
        is_test=is_test
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=cnn_bilstm_collate_fn
    )

    model.eval()
    predictions = {}

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            pred_ids = torch.argmax(logits, dim=1).cpu().numpy().tolist()

            for claim_id, pred_id, evidence_ids in zip(batch["claim_ids"], pred_ids, batch["evidence_ids"]):
              predictions[claim_id] = {
                  "claim_text": claims_json[claim_id]["claim_text"],
                  "claim_label": ID2LABEL[pred_id],
                  "evidences": evidence_ids[:max_evidence]
              }

    save_json(predictions, output_path)
    print("Saved predictions to:", output_path)

    return predictions

## Generate dev predictions

In [42]:
CNN_BILSTM_DEV_PRED_PATH = OUTPUT_DIR / "dev_predictions_cnn_bilstm_multihead_4evi.json"

dev_predictions_cnn_bilstm = predict_claims_cnn_bilstm(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model=cnn_bilstm_multihead_model,
    vocab=vocab,
    output_path=CNN_BILSTM_DEV_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=False,
    device=device
)

  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/dev_predictions_cnn_bilstm_multihead_4evi.json


## predict test claims

In [43]:
test_claims = load_json(TEST_PATH)

TEST_PRED_PATH = OUTPUT_DIR / "test_predictions_cnn_bilstm_multihead.json"

test_predictions = predict_claims_cnn_bilstm(
    claims_json=test_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model=cnn_bilstm_multihead_model,
    vocab=vocab,
    output_path=TEST_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=True,
    device=device
)

  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/test_predictions_cnn_bilstm_multihead.json


## eval.py command line

In [44]:
import os

# Change the current working directory to the NLP folder in Google Drive
os.chdir('/content/drive/MyDrive/NLP/')

In [45]:
!python eval.py --predictions outputs/dev_predictions_cnn_bilstm_multihead_4evi.json --groundtruth data/dev-claims.json


Evidence Retrieval F-score (F)    = 0.1509018759018759
Claim Classification Accuracy (A) = 0.44805194805194803
Harmonic Mean of F and A          = 0.22576658419578374


# CNN+BiLSTM - multihead (balanced class)

## Encode text

In [54]:
def encode_tokens(tokens, vocab, max_len=512):
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    return ids[:max_len]


def encode_claim_evidence(claim_text, evidence_text, vocab, max_len=512):
    tokens = (
        ["<CLAIM>"]
        + simple_tokenise(claim_text)
        + ["<EVIDENCE>"]
        + simple_tokenise(evidence_text)
    )

    return encode_tokens(tokens, vocab, max_len=max_len)

## CNN + BiLSTM dataset

In [55]:
class CNNBiLSTMDataset(Dataset):
    def __init__(
        self,
        claims_json,
        evidence_corpus,
        vocab,
        max_len=512,
        max_evidence=5,
        use_gold_evidence=True,
        retriever=None,
        retrieval_top_k=5,
        is_test=False
    ):
        self.items = list(claims_json.items())
        self.evidence_corpus = evidence_corpus
        self.vocab = vocab
        self.max_len = max_len
        self.max_evidence = max_evidence
        self.use_gold_evidence = use_gold_evidence
        self.retriever = retriever
        self.retrieval_top_k = retrieval_top_k
        self.is_test = is_test

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        claim_id, instance = self.items[idx]
        claim_text = instance["claim_text"]

        if self.use_gold_evidence and not self.is_test:
            evidence_ids = instance.get("evidences", [])
        else:
            evidence_ids = self.retriever.retrieve(
                claim_text,
                top_k=self.retrieval_top_k
            )

        evidence_text = concatenate_evidence(
            evidence_ids=evidence_ids,
            evidence_corpus=self.evidence_corpus,
            max_evidence=self.max_evidence
        )

        input_ids = encode_claim_evidence(
            claim_text=claim_text,
            evidence_text=evidence_text,
            vocab=self.vocab,
            max_len=self.max_len
        )

        item = {
            "claim_id": claim_id,
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "evidence_ids": evidence_ids
        }

        if not self.is_test:
            item["label"] = torch.tensor(
                LABEL2ID[instance["claim_label"]],
                dtype=torch.long
            )

        return item


def cnn_bilstm_collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    attention_mask = (input_ids != vocab["<PAD>"]).long()

    output = {
        "claim_ids": [item["claim_id"] for item in batch],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "evidence_ids": [item["evidence_ids"] for item in batch]
    }

    if "label" in batch[0]:
        output["labels"] = torch.stack([item["label"] for item in batch])

    return output

## Model Design

In [56]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, hidden_states, attention_mask=None):
        scores = self.attention(hidden_states).squeeze(-1)

        if attention_mask is not None:
            attention_mask = attention_mask.bool()
            scores = scores.masked_fill(~attention_mask, -1e9)

        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(hidden_states * weights.unsqueeze(-1), dim=1)

        return pooled


class CNNBiLSTMMultiheadClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx
        )

        self.embedding_dropout = nn.Dropout(dropout)

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=cnn_channels,
                kernel_size=k,
                padding=k // 2
            )
            for k in kernel_sizes
        ])

        cnn_output_dim = cnn_channels * len(kernel_sizes)

        self.bilstm = nn.LSTM(
            input_size=cnn_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        lstm_output_dim = lstm_hidden_dim * 2

        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=lstm_output_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True
        )
        self.layer_norm = nn.LayerNorm(lstm_output_dim)

        self.attention_pooling = AttentionPooling(lstm_output_dim)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, num_labels)
        )

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        # [batch, seq_len, embedding_dim] -> [batch, embedding_dim, seq_len]
        conv_input = embedded.transpose(1, 2)

        conv_outputs = []

        for conv in self.convs:
            x = F.relu(conv(conv_input))
            conv_outputs.append(x)

        # [batch, cnn_channels * num_kernels, seq_len]
        conv_output = torch.cat(conv_outputs, dim=1)

        # [batch, seq_len, cnn_channels * num_kernels]
        lstm_input = conv_output.transpose(1, 2)

        lstm_output, _ = self.bilstm(lstm_input)

        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = attention_mask == 0

        attn_output, _ = self.multihead_attn(
            query=lstm_output,
            key=lstm_output,
            value=lstm_output,
            key_padding_mask=key_padding_mask
        )

        attn_output = self.layer_norm(attn_output + lstm_output)

        pooled_output = self.attention_pooling(
            attn_output,
            attention_mask=attention_mask
        )

        pooled_output = self.dropout(pooled_output)

        logits = self.classifier(pooled_output)

        return logits

## Evaluation function

In [57]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

def evaluate_cnn_bilstm(model, dataloader, device="cpu"):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device).long()

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    per_class_f1 = f1_score(
        all_labels,
        all_preds,
        average=None,
        labels=list(range(4)),
        zero_division=0
    )

    print("Dev classification accuracy:", round(acc, 4))
    print("Dev macro F1:", round(macro_f1, 4))
    print("Dev weighted F1:", round(weighted_f1, 4))

    print("\nPer-class F1:")
    for i, score in enumerate(per_class_f1):
        print(f"{ID2LABEL[i]}: {score:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            all_labels,
            all_preds,
            labels=list(range(4)),
            target_names=[ID2LABEL[i] for i in range(4)],
            zero_division=0
        )
    )

    print("\nConfusion matrix:")
    print(confusion_matrix(all_labels, all_preds, labels=list(range(4))))

    return acc, macro_f1, weighted_f1

## Train CNN + BiLSTM

In [58]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight

def get_class_weights_from_dataset(dataset, num_labels, device):
    labels = []

    for item in dataset:
        label = item["label"] if "label" in item else item["labels"]
        labels.append(int(label))

    labels = np.array(labels)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_labels),
        y=labels
    )

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float
    ).to(device)

    return class_weights

In [59]:
from sklearn.utils.class_weight import compute_class_weight

def train_cnn_bilstm_multikernel_multihead_balanced(
    train_claims,
    dev_claims,
    evidence_corpus,
    vocab,
    epochs=5,
    batch_size=32,
    lr=1e-3,
    max_len=256,
    max_evidence=4,
    device="cpu"
):
    set_seed(42)

    train_dataset = CNNBiLSTMDataset(
        claims_json=train_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    dev_dataset = CNNBiLSTMDataset(
        claims_json=dev_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    g = torch.Generator()
    g.manual_seed(42)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=cnn_bilstm_collate_fn,
        generator=g
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=cnn_bilstm_collate_fn
    )

    class_weights = get_class_weights_from_dataset(
    dataset=train_dataset,
    num_labels=4,
    device=device
    )

    model = CNNBiLSTMMultiheadClassifier(
        vocab_size=len(vocab),
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=vocab["<PAD>"]
    ).to(device)

    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )


    print("Class weights:", class_weights)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_macro_f1 = 0.0
    best_state_dict = None

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimiser.zero_grad()

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimiser.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print("Training loss:", round(avg_loss, 4))

        dev_acc, dev_macro_f1, dev_weighted_f1 = evaluate_cnn_bilstm(
            model,
            dev_loader,
            device
        )

        print("Dev accuracy with gold evidence:", round(dev_acc, 4))
        print("Dev macro F1 with gold evidence:", round(dev_macro_f1, 4))
        print("Dev weighted F1 with gold evidence:", round(dev_weighted_f1, 4))

        if dev_macro_f1 > best_macro_f1:
            best_macro_f1 = dev_macro_f1
            best_state_dict = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }
            print("New best model saved in memory.")

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model

In [68]:
cnn_bilstm_multihead_model = train_cnn_bilstm_multikernel_multihead_balanced(
    train_claims=train_claims,
    dev_claims=dev_claims,
    evidence_corpus=evidence_corpus,
    vocab=vocab,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    max_len=256,
    max_evidence=4,
    device=device
)

Class weights: tensor([0.5915, 1.5427, 0.7953, 2.4758])

Epoch 1/10


  0%|          | 0/39 [00:00<?, ?it/s]

Training loss: 1.416


  0%|          | 0/5 [00:00<?, ?it/s]

Dev classification accuracy: 0.1753
Dev macro F1: 0.0746
Dev weighted F1: 0.0523

Per-class F1:
SUPPORTS: 0.0000
REFUTES: 0.2983
NOT_ENOUGH_INFO: 0.0000
DISPUTED: 0.0000

Classification report:
                 precision    recall  f1-score   support

       SUPPORTS       0.00      0.00      0.00        68
        REFUTES       0.18      1.00      0.30        27
NOT_ENOUGH_INFO       0.00      0.00      0.00        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.18       154
      macro avg       0.04      0.25      0.07       154
   weighted avg       0.03      0.18      0.05       154


Confusion matrix:
[[ 0 68  0  0]
 [ 0 27  0  0]
 [ 0 41  0  0]
 [ 0 18  0  0]]
Dev accuracy with gold evidence: 0.1753
Dev macro F1 with gold evidence: 0.0746
Dev weighted F1 with gold evidence: 0.0523
New best model saved in memory.

Epoch 2/10


...

Dev classification accuracy: 0.4481
Dev macro F1: 0.4155
Dev weighted F1: 0.4346

Per-class F1:
SUPPORTS: 0.3800
REFUTES: 0.4179
NOT_ENOUGH_INFO: 0.6200
DISPUTED: 0.2439

Classification report:
                 precision    recall  f1-score   support

       SUPPORTS       0.59      0.28      0.38        68
        REFUTES       0.35      0.52      0.42        27
NOT_ENOUGH_INFO       0.53      0.76      0.62        41
       DISPUTED       0.22      0.28      0.24        18

       accuracy                           0.45       154
      macro avg       0.42      0.46      0.42       154
   weighted avg       0.49      0.45      0.43       154


Confusion matrix:
[[19 17 20 12]
 [ 3 14  4  6]
 [ 6  4 31  0]
 [ 4  5  4  5]]
Dev accuracy with gold evidence: 0.4481
Dev macro F1 with gold evidence: 0.4155
Dev weighted F1 with gold evidence: 0.4346
New best model saved in memory.

Epoch 10/10


  0%|          | 0/39 [00:00<?, ?it/s]

Training loss: 1.0177


  0%|          | 0/5 [00:00<?, ?it/s]

Dev classification accuracy: 0.3442
Dev macro F1: 0.2169
Dev weighted F1: 0.272

Per-class F1:
SUPPORTS: 0.2970
REFUTES: 0.0000
NOT_ENOUGH_INFO: 0.4966
DISPUTED: 0.0741

Classification report:
                 precision    recall  f1-score   support

       SUPPORTS       0.45      0.22      0.30        68
        REFUTES       0.00      0.00      0.00        27
NOT_ENOUGH_INFO       0.34      0.90      0.50        41
       DISPUTED       0.11      0.06      0.07        18

       accuracy                           0.34       154
      macro avg       0.23      0.29      0.22       154
   weighted avg       0.30      0.34      0.27       154


Confusion matrix:
[[15  3 46  4]
 [ 8  0 15  4]
 [ 4  0 37  0]
 [ 6  1 10  1]]
Dev accuracy with gold evidence: 0.3442
Dev macro F1 with gold evidence: 0.2169
Dev weighted F1 with gold evidence: 0.272


## Predict with retrieved evidence

In [69]:
def predict_claims_cnn_bilstm(
    claims_json,
    evidence_corpus,
    retriever,
    model,
    vocab,
    output_path,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=False,
    device="cpu"
):
    dataset = CNNBiLSTMDataset(
        claims_json=claims_json,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=False,
        retriever=retriever,
        retrieval_top_k=retrieval_top_k,
        is_test=is_test
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=cnn_bilstm_collate_fn
    )

    model.eval()
    predictions = {}

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            pred_ids = torch.argmax(logits, dim=1).cpu().numpy().tolist()

            for claim_id, pred_id, evidence_ids in zip(batch["claim_ids"], pred_ids, batch["evidence_ids"]):
              predictions[claim_id] = {
                  "claim_text": claims_json[claim_id]["claim_text"],
                  "claim_label": ID2LABEL[pred_id],
                  "evidences": evidence_ids[:max_evidence]
              }

    save_json(predictions, output_path)
    print("Saved predictions to:", output_path)

    return predictions

## Generate dev predictions

In [70]:
def evaluate_dev_predictions_classification_f1(dev_claims, predictions):
    gold_labels = []
    pred_labels = []

    for claim_id, gold_instance in dev_claims.items():
        if claim_id not in predictions:
            continue

        gold_label = gold_instance["claim_label"]
        pred_label = predictions[claim_id]["claim_label"]

        gold_labels.append(LABEL2ID[gold_label])
        pred_labels.append(LABEL2ID[pred_label])

    acc = accuracy_score(gold_labels, pred_labels)

    macro_f1 = f1_score(
        gold_labels,
        pred_labels,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        gold_labels,
        pred_labels,
        average="weighted",
        zero_division=0
    )

    per_class_f1 = f1_score(
        gold_labels,
        pred_labels,
        average=None,
        labels=list(range(4)),
        zero_division=0
    )

    print("Dev classification accuracy:", round(acc, 4))
    print("Dev classification macro F1:", round(macro_f1, 4))
    print("Dev classification weighted F1:", round(weighted_f1, 4))

    print("\nPer-class F1:")
    for i, score in enumerate(per_class_f1):
        print(f"{ID2LABEL[i]}: {score:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            gold_labels,
            pred_labels,
            labels=list(range(4)),
            target_names=[ID2LABEL[i] for i in range(4)],
            zero_division=0
        )
    )

    print("\nConfusion matrix:")
    print(confusion_matrix(gold_labels, pred_labels, labels=list(range(4))))

    return acc, macro_f1, weighted_f1

In [71]:
CNN_BILSTM_DEV_PRED_PATH = OUTPUT_DIR / "dev_predictions_cnn_bilstm_multihead_bal_4evi.json"

dev_predictions_cnn_bilstm = predict_claims_cnn_bilstm(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model=cnn_bilstm_multihead_model,
    vocab=vocab,
    output_path=CNN_BILSTM_DEV_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=False,
    device=device
)

  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/dev_predictions_cnn_bilstm_multihead_bal_4evi.json


In [72]:
dev_cls_acc, dev_cls_macro_f1, dev_cls_weighted_f1 = evaluate_dev_predictions_classification_f1(
    dev_claims=dev_claims,
    predictions=dev_predictions_cnn_bilstm
)

Dev classification accuracy: 0.3247
Dev classification macro F1: 0.3016
Dev classification weighted F1: 0.3169

Per-class F1:
SUPPORTS: 0.2772
REFUTES: 0.2951
NOT_ENOUGH_INFO: 0.4600
DISPUTED: 0.1739

Classification report:
                 precision    recall  f1-score   support

       SUPPORTS       0.42      0.21      0.28        68
        REFUTES       0.26      0.33      0.30        27
NOT_ENOUGH_INFO       0.39      0.56      0.46        41
       DISPUTED       0.14      0.22      0.17        18

       accuracy                           0.32       154
      macro avg       0.31      0.33      0.30       154
   weighted avg       0.35      0.32      0.32       154


Confusion matrix:
[[14 14 25 15]
 [ 6  9  8  4]
 [ 9  4 23  5]
 [ 4  7  3  4]]


## predict test claims

In [73]:
test_claims = load_json(TEST_PATH)

TEST_PRED_PATH = OUTPUT_DIR / "test_predictions_cnn_bilstm_multihead_bal.json"

test_predictions = predict_claims_cnn_bilstm(
    claims_json=test_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model=cnn_bilstm_multihead_model,
    vocab=vocab,
    output_path=TEST_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=True,
    device=device
)

  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/test_predictions_cnn_bilstm_multihead_bal.json


## eval.py command line

In [74]:
import os

# Change the current working directory to the NLP folder in Google Drive
os.chdir('/content/drive/MyDrive/NLP/')

In [75]:
!python eval.py --predictions outputs/dev_predictions_cnn_bilstm_multihead_bal_4evi.json --groundtruth data/dev-claims.json


Evidence Retrieval F-score (F)    = 0.1509018759018759
Claim Classification Accuracy (A) = 0.3246753246753247
Harmonic Mean of F and A          = 0.2060406406913272
